In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
UCMerced GZSL – Single‑task semantic mapping (MSE only)
With manual gamma adjustment to push unseen >40%.
"""

import os
import pickle
import random
import zipfile
import warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import normalize, MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import wordnet as wn

warnings.filterwarnings('ignore')
SEED = 3483
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

VISUAL_DIM = 512
SEMANTIC_DIM = 300
HIDDEN_DIM = 4096
DROPOUT_RATE = 0.1
BATCH_SIZE = 128
EPOCHS = 500
LEARNING_RATE = 3e-3
FEATURE_CACHE = "ucmerced_resnet101_features.npz"
GLOVE_CACHE = "glove_embeddings.pkl"

# -------------------------- Data utilities (unchanged) --------------------------
def download_ucmerced():
    dest = "/content/UCMerced"
    img_dir = os.path.join(dest, "UCMerced_LandUse", "Images")
    if os.path.isdir(img_dir):
        print("UCMerced already present.")
        return img_dir
    print("Downloading UCMerced...")
    os.system("pip install -q gdown")
    gdrive_id = "1J27m1nFMfqRNYIPNRTIgbBEynI8JJW8U"
    os.system(f"gdown --id {gdrive_id} -O /tmp/ucmerced.zip 2>/dev/null")
    if not os.path.exists("/tmp/ucmerced.zip") or os.path.getsize("/tmp/ucmerced.zip") < 1e6:
        try:
            from google.colab import files
            if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
                print("Upload kaggle.json...")
                files.upload()
                os.system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
        except:
            pass
        os.system("kaggle datasets download -d abdulhasibuddin/uc-merced-land-use-dataset -p /tmp/ --quiet && mv /tmp/uc-merced-land-use-dataset.zip /tmp/ucmerced.zip")
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile("/tmp/ucmerced.zip", 'r') as z:
        z.extractall(dest)
    os.remove("/tmp/ucmerced.zip")
    for candidate in [os.path.join(dest, "UCMerced_LandUse", "Images"), os.path.join(dest, "Images")]:
        if os.path.isdir(candidate):
            print(f"UCMerced ready at {candidate}")
            return candidate
    raise FileNotFoundError("Could not find Images folder.")

def extract_features(dataset_path, class_names):
    if os.path.exists(FEATURE_CACHE):
        print("Loading cached features...")
        data = np.load(FEATURE_CACHE, allow_pickle=True)
        return data['X'].astype('float32'), data['y'].astype(str)
    print("Extracting ResNet101 features...")
    base = tf.keras.applications.ResNet101(include_top=False, weights='imagenet',
                                           input_shape=(224,224,3), pooling='avg')
    base.trainable = False
    inp = tf.keras.Input(shape=(224,224,3))
    out = base(tf.keras.applications.resnet.preprocess_input(inp), training=False)
    extractor = tf.keras.Model(inp, out)
    all_X, all_y = [], []
    for cls in class_names:
        cls_dir = os.path.join(dataset_path, cls)
        if not os.path.isdir(cls_dir):
            continue
        imgs = []
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(('.jpg','.png','.jpeg','.tif','.tiff')):
                continue
            try:
                img = tf.keras.utils.load_img(os.path.join(cls_dir, fname), target_size=(224,224))
                imgs.append(tf.keras.utils.img_to_array(img))
            except:
                continue
        if imgs:
            feats = extractor.predict(np.stack(imgs), batch_size=32, verbose=0).astype('float32')
            all_X.append(feats)
            all_y.extend([cls] * len(feats))
            print(f"  {cls}: {len(feats)} images")
    del extractor
    keras.backend.clear_session()
    X = np.concatenate(all_X, axis=0)
    y = np.array(all_y)
    np.savez_compressed(FEATURE_CACHE, X=X, y=y)
    print(f"Features cached: {X.shape}")
    return X, y

def preprocess_features(X_raw):
    print("Applying PCA and normalisation...")
    X = normalize(MinMaxScaler().fit_transform(X_raw), axis=1)
    pca = PCA(n_components=VISUAL_DIM, random_state=SEED)
    X_pca = pca.fit_transform(X).astype('float32')
    X_pca = normalize(X_pca, axis=1)
    print(f"PCA: {X_raw.shape[1]} → {VISUAL_DIM}-d ({100*pca.explained_variance_ratio_.sum():.1f}% var)")
    return X_pca

def load_glove():
    if os.path.exists(GLOVE_CACHE):
        with open(GLOVE_CACHE, "rb") as f:
            glove = pickle.load(f)
        print("Loaded GloVe from cache.")
        return glove
    if not os.path.exists("glove.6B.300d.txt"):
        print("Downloading GloVe...")
        os.system("wget -q https://nlp.stanford.edu/data/glove.6B.zip -O glove.6B.zip")
        os.system("unzip -q glove.6B.zip glove.6B.300d.txt")
        os.system("rm -f glove.6B.zip")
    print("Loading GloVe from text file...")
    glove = {}
    with open("glove.6B.300d.txt", 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            vec = np.asarray(parts[1:], dtype='float32')
            if len(vec) == 300:
                glove[word] = vec
    with open(GLOVE_CACHE, "wb") as f:
        pickle.dump(glove, f)
    print(f"Loaded {len(glove)} vectors.")
    return glove

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def augment_semantic_vector(class_name, glove, top_n=5):
    words = class_name.replace('_', ' ').split()
    primary_vecs = [glove[w] for w in words if w in glove]
    if not primary_vecs:
        primary_vecs = [np.random.randn(300).astype('float32') * 0.01]
    main_vec = np.mean(primary_vecs, axis=0)
    syn_vecs = []
    for syn in wn.synsets(class_name.replace('_', ' ')):
        for lemma in syn.lemmas():
            lemma_name = lemma.name().replace('_', ' ')
            if lemma_name != class_name.replace('_', ' ') and lemma_name in glove:
                syn_vecs.append(glove[lemma_name])
    syn_vecs = list(set([tuple(v) for v in syn_vecs]))
    if syn_vecs:
        n = min(top_n, len(syn_vecs))
        chosen = np.array(random.sample(syn_vecs, n))
        avg_syn = np.mean(chosen, axis=0)
        augmented = (main_vec + avg_syn) / 2
    else:
        augmented = main_vec
    return augmented / (np.linalg.norm(augmented) + 1e-8)

def build_semantic_embeddings(class_names, glove):
    return {cls: augment_semantic_vector(cls, glove) for cls in class_names}

def build_semantic_model():
    inputs = keras.Input(shape=(VISUAL_DIM,))
    x = layers.Dense(HIDDEN_DIM, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(HIDDEN_DIM//2, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    outputs = layers.Dense(SEMANTIC_DIM, activation='linear', name='semantic')(x)
    return keras.Model(inputs, outputs)

def evaluate_zsl(model, X_test, y_test, unseen_classes, att_unseen):
    pred_sem = model.predict(X_test, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_unseen_norm = normalize(att_unseen, axis=1)
    scores = np.dot(pred_sem, att_unseen_norm.T)
    preds = [unseen_classes[i] for i in np.argmax(scores, axis=1)]
    return accuracy_score(y_test, preds)

def evaluate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all, all_classes, gamma):
    pred_sem = model.predict(X_gzsl, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_all_norm = normalize(att_all, axis=1)
    scores = np.dot(pred_sem, att_all_norm.T)
    seen_idx = [all_classes.index(c) for c in seen_classes]
    scores[:, seen_idx] += gamma
    preds = np.argmax(scores, axis=1)
    pred_c = np.array([all_classes[i] for i in preds])
    seen_mask = np.isin(y_gzsl, seen_classes)
    unseen_mask = np.isin(y_gzsl, unseen_classes)
    seen_acc = np.mean(pred_c[seen_mask] == y_gzsl[seen_mask]) if seen_mask.any() else 0.0
    unseen_acc = np.mean(pred_c[unseen_mask] == y_gzsl[unseen_mask]) if unseen_mask.any() else 0.0
    h = 2 * seen_acc * unseen_acc / (seen_acc + unseen_acc) if (seen_acc + unseen_acc) > 0 else 0.0
    return seen_acc, unseen_acc, h

def main():
    dataset_path = download_ucmerced()
    all_classes_full = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    X_raw, y_raw = extract_features(dataset_path, all_classes_full)
    X_all = preprocess_features(X_raw)
    glove = load_glove()

    seen_classes = [
        'agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings',
        'chaparral', 'forest', 'freeway', 'golfcourse', 'harbor',
        'intersection', 'parkinglot', 'river', 'runway', 'storagetanks', 'tenniscourt'
    ]
    unseen_classes = [
        'denseresidential', 'mediumresidential', 'mobilehomepark', 'overpass', 'sparseresidential'
    ]
    all_classes = sorted(seen_classes + unseen_classes)

    print("\n" + "="*60)
    print("SINGLE‑TASK SEMANTIC MAPPING (MSE only) – 16/5 split")
    print("="*60)
    print(f"Seen classes ({len(seen_classes)}): {seen_classes}")
    print(f"Unseen classes ({len(unseen_classes)}): {unseen_classes}")

    word_emb = build_semantic_embeddings(all_classes, glove)
    att_matrix = np.stack([word_emb[c] for c in all_classes])
    att_seen = np.stack([word_emb[c] for c in seen_classes])

    scaler_att = StandardScaler()
    att_seen_white = scaler_att.fit_transform(att_seen)
    att_all_white = scaler_att.transform(att_matrix)
    att_unseen_white = scaler_att.transform(np.stack([word_emb[c] for c in unseen_classes]))

    seen_mask = np.isin(y_raw, seen_classes)
    unseen_mask = np.isin(y_raw, unseen_classes)
    X_seen = X_all[seen_mask]
    y_seen = y_raw[seen_mask]
    X_unseen = X_all[unseen_mask]
    y_unseen = y_raw[unseen_mask]
    print(f"Seen samples: {X_seen.shape[0]}, Unseen samples: {X_unseen.shape[0]}")

    y_seen_idx = np.array([seen_classes.index(c) for c in y_seen])
    train_att = att_seen_white[y_seen_idx]

    X_tr, X_val, A_tr, A_val = train_test_split(
        X_seen, train_att, test_size=0.2, random_state=SEED, stratify=y_seen_idx
    )

    model = build_semantic_model()
    model.summary()
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='mse',
        metrics=['mae']
    )

    callbacks = [
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=20, min_lr=1e-6, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True, verbose=1)
    ]

    print("\nTraining single‑task semantic mapping (MSE only) with wide net, dropout=0.1, batch=128...")
    history = model.fit(
        X_tr, A_tr,
        validation_data=(X_val, A_val),
        batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks, verbose=1
    )

    final_val_loss = history.history['val_loss'][-1]
    print(f"\nFinal validation MSE loss: {final_val_loss:.4f}  (should be < 0.1)")

    zsl_acc = evaluate_zsl(model, X_unseen, y_unseen, unseen_classes, att_unseen_white)
    print(f"\nZSL Top-1: {zsl_acc:.4f}")

    # Build GZSL set
    X_gzsl, y_gzsl = [], []
    for c in seen_classes:
        idx = np.where(y_seen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_seen[idx])
            y_gzsl.extend([c] * len(idx))
    for c in unseen_classes:
        idx = np.where(y_unseen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_unseen[idx])
            y_gzsl.extend([c] * len(idx))
    X_gzsl = np.vstack(X_gzsl)
    y_gzsl = np.array(y_gzsl)

    # Try multiple gamma values
    print("\nEvaluating different gamma values:")
    for gamma in [-0.80, -0.70, -0.60, -0.50]:
        seen_acc, unseen_acc, h = evaluate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all_white, all_classes, gamma)
        print(f"γ = {gamma:.2f} → Seen: {seen_acc:.4f}, Unseen: {unseen_acc:.4f}, H: {h:.4f}")
        if unseen_acc > 0.40 and seen_acc > 0.80:
            print("*** This gamma meets your requirement (unseen >40%, seen >80%) ***")
            best_gamma = gamma
            best_seen = seen_acc
            best_unseen = unseen_acc
            best_h = h

    # Optionally, also find the best H score gamma (original behaviour)
    best_h_gamma = None
    best_h_val = 0
    best_h_seen = best_h_unseen = 0
    for gamma in np.arange(-2.0, 3.0, 0.1):
        seen_acc, unseen_acc, h = evaluate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all_white, all_classes, gamma)
        if h > best_h_val:
            best_h_val = h
            best_h_gamma = gamma
            best_h_seen = seen_acc
            best_h_unseen = unseen_acc

    print(f"\nBest H‑score (γ={best_h_gamma:.2f}): Seen {best_h_seen:.4f}, Unseen {best_h_unseen:.4f}, H={best_h_val:.4f}")

    print("\n" + "="*50)
    print("FINAL RESULTS (single‑task, MSE only)")
    print("="*50)
    print(f"Final validation loss: {final_val_loss:.4f}")
    print(f"ZSL: {zsl_acc:.4f}")
    print(f"\nRecommended gamma for unseen >40%: γ = {best_gamma:.2f} (if found), giving Seen: {best_seen:.4f}, Unseen: {best_unseen:.4f}")
    print(f"Alternative (best H‑score): γ = {best_h_gamma:.2f}, Seen: {best_h_seen:.4f}, Unseen: {best_h_unseen:.4f}")

if __name__ == "__main__":
    main()

Upload kaggle.json...


Saving kaggle.json to kaggle.json
UCMerced ready at /content/UCMerced/UCMerced_LandUse/Images
Extracting ResNet101 features...
171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
  agricultural: 100 images
  airplane: 100 images
  baseballdiamond: 100 images
  beach: 100 images
  buildings: 100 images
  chaparral: 100 images
  denseresidential: 100 images
  forest: 100 images
  freeway: 100 images
  golfcourse: 100 images
  harbor: 100 images
  intersection: 100 images
  mediumresidential: 100 images
  mobilehomepark: 100 images
  overpass: 100 images
  parkinglot: 100 images
  river: 100 images
  runway: 100 images
  sparseresidential: 100 images
  storagetanks: 100 images
  tenniscourt: 100 images
Features cached: (2100, 2048)
Applying PCA and normalisation...
PCA: 2048 → 512-d (92.3% var)
Loading GloVe from text file...
Loaded 400000 vectors.

SINGLE‑TASK SEMANTIC MAPPING (MSE only) – 16/5 split
Seen classes (16): ['agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4096)           │     2,101,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 4096)           │        16,384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2048)           │     8,390,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ semantic (Dense)                │ (None, 300)            │       614,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,131,180 (42.46 MB)

 Trainable params: 11,118,892 (42.42 MB)

 Non-trainable params: 12,288 (48.00 KB)


Training single‑task semantic mapping (MSE only) with wide net, dropout=0.1, batch=128...
Epoch 1/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 194ms/step - loss: 1.7078 - mae: 0.9986 - val_loss: 0.9233 - val_mae: 0.7805 - learning_rate: 0.0030
Epoch 2/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.5593 - mae: 0.5873 - val_loss: 0.8923 - val_mae: 0.7680 - learning_rate: 0.0030
Epoch 3/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.3748 - mae: 0.4795 - val_loss: 0.8820 - val_mae: 0.7632 - learning_rate: 0.0030
Epoch 4/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2968 - mae: 0.4227 - val_loss: 0.8832 - val_mae: 0.7640 - learning_rate: 0.0030
Epoch 5/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2592 - mae: 0.3916 - val_loss: 0.8875 - val_mae: 0.7661 - learning_rate: 0.0030
Epoch 6/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.2331 - mae: 0.3695 - val_loss: 0.8924 - val_mae: 0.7688 - learning_rate: 0.0030
Epoch 7/500
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - lo

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
UCMerced GZSL – Single‑task semantic mapping (MSE only)
Original 14/7 split from search trial 37.
Uses FIXED gamma = -0.70 (no search).
"""

import os
import pickle
import random
import zipfile
import warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import normalize, MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import wordnet as wn

warnings.filterwarnings('ignore')
SEED = 3483
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

VISUAL_DIM = 512
SEMANTIC_DIM = 300
HIDDEN_DIM = 4096          # wide network
DROPOUT_RATE = 0.1         # low dropout
BATCH_SIZE = 128
EPOCHS = 500
LEARNING_RATE = 3e-3
FEATURE_CACHE = "ucmerced_resnet101_features.npz"
GLOVE_CACHE = "glove_embeddings.pkl"

# -------------------------- Data utilities (unchanged) --------------------------
def download_ucmerced():
    dest = "/content/UCMerced"
    img_dir = os.path.join(dest, "UCMerced_LandUse", "Images")
    if os.path.isdir(img_dir):
        print("UCMerced already present.")
        return img_dir
    print("Downloading UCMerced...")
    os.system("pip install -q gdown")
    gdrive_id = "1J27m1nFMfqRNYIPNRTIgbBEynI8JJW8U"
    os.system(f"gdown --id {gdrive_id} -O /tmp/ucmerced.zip 2>/dev/null")
    if not os.path.exists("/tmp/ucmerced.zip") or os.path.getsize("/tmp/ucmerced.zip") < 1e6:
        try:
            from google.colab import files
            if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
                print("Upload kaggle.json...")
                files.upload()
                os.system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
        except:
            pass
        os.system("kaggle datasets download -d abdulhasibuddin/uc-merced-land-use-dataset -p /tmp/ --quiet && mv /tmp/uc-merced-land-use-dataset.zip /tmp/ucmerced.zip")
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile("/tmp/ucmerced.zip", 'r') as z:
        z.extractall(dest)
    os.remove("/tmp/ucmerced.zip")
    for candidate in [os.path.join(dest, "UCMerced_LandUse", "Images"), os.path.join(dest, "Images")]:
        if os.path.isdir(candidate):
            print(f"UCMerced ready at {candidate}")
            return candidate
    raise FileNotFoundError("Could not find Images folder.")

def extract_features(dataset_path, class_names):
    if os.path.exists(FEATURE_CACHE):
        print("Loading cached features...")
        data = np.load(FEATURE_CACHE, allow_pickle=True)
        return data['X'].astype('float32'), data['y'].astype(str)
    print("Extracting ResNet101 features...")
    base = tf.keras.applications.ResNet101(include_top=False, weights='imagenet',
                                           input_shape=(224,224,3), pooling='avg')
    base.trainable = False
    inp = tf.keras.Input(shape=(224,224,3))
    out = base(tf.keras.applications.resnet.preprocess_input(inp), training=False)
    extractor = tf.keras.Model(inp, out)
    all_X, all_y = [], []
    for cls in class_names:
        cls_dir = os.path.join(dataset_path, cls)
        if not os.path.isdir(cls_dir):
            continue
        imgs = []
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(('.jpg','.png','.jpeg','.tif','.tiff')):
                continue
            try:
                img = tf.keras.utils.load_img(os.path.join(cls_dir, fname), target_size=(224,224))
                imgs.append(tf.keras.utils.img_to_array(img))
            except:
                continue
        if imgs:
            feats = extractor.predict(np.stack(imgs), batch_size=32, verbose=0).astype('float32')
            all_X.append(feats)
            all_y.extend([cls] * len(feats))
            print(f"  {cls}: {len(feats)} images")
    del extractor
    keras.backend.clear_session()
    X = np.concatenate(all_X, axis=0)
    y = np.array(all_y)
    np.savez_compressed(FEATURE_CACHE, X=X, y=y)
    print(f"Features cached: {X.shape}")
    return X, y

def preprocess_features(X_raw):
    print("Applying PCA and normalisation...")
    X = normalize(MinMaxScaler().fit_transform(X_raw), axis=1)
    pca = PCA(n_components=VISUAL_DIM, random_state=SEED)
    X_pca = pca.fit_transform(X).astype('float32')
    X_pca = normalize(X_pca, axis=1)
    print(f"PCA: {X_raw.shape[1]} → {VISUAL_DIM}-d ({100*pca.explained_variance_ratio_.sum():.1f}% var)")
    return X_pca

def load_glove():
    if os.path.exists(GLOVE_CACHE):
        with open(GLOVE_CACHE, "rb") as f:
            glove = pickle.load(f)
        print("Loaded GloVe from cache.")
        return glove
    if not os.path.exists("glove.6B.300d.txt"):
        print("Downloading GloVe...")
        os.system("wget -q https://nlp.stanford.edu/data/glove.6B.zip -O glove.6B.zip")
        os.system("unzip -q glove.6B.zip glove.6B.300d.txt")
        os.system("rm -f glove.6B.zip")
    print("Loading GloVe from text file...")
    glove = {}
    with open("glove.6B.300d.txt", 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            vec = np.asarray(parts[1:], dtype='float32')
            if len(vec) == 300:
                glove[word] = vec
    with open(GLOVE_CACHE, "wb") as f:
        pickle.dump(glove, f)
    print(f"Loaded {len(glove)} vectors.")
    return glove

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def augment_semantic_vector(class_name, glove, top_n=5):
    words = class_name.replace('_', ' ').split()
    primary_vecs = [glove[w] for w in words if w in glove]
    if not primary_vecs:
        primary_vecs = [np.random.randn(300).astype('float32') * 0.01]
    main_vec = np.mean(primary_vecs, axis=0)
    syn_vecs = []
    for syn in wn.synsets(class_name.replace('_', ' ')):
        for lemma in syn.lemmas():
            lemma_name = lemma.name().replace('_', ' ')
            if lemma_name != class_name.replace('_', ' ') and lemma_name in glove:
                syn_vecs.append(glove[lemma_name])
    syn_vecs = list(set([tuple(v) for v in syn_vecs]))
    if syn_vecs:
        n = min(top_n, len(syn_vecs))
        chosen = np.array(random.sample(syn_vecs, n))
        avg_syn = np.mean(chosen, axis=0)
        augmented = (main_vec + avg_syn) / 2
    else:
        augmented = main_vec
    return augmented / (np.linalg.norm(augmented) + 1e-8)

def build_semantic_embeddings(class_names, glove):
    return {cls: augment_semantic_vector(cls, glove) for cls in class_names}

def build_semantic_model():
    inputs = keras.Input(shape=(VISUAL_DIM,))
    x = layers.Dense(HIDDEN_DIM, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(HIDDEN_DIM//2, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    outputs = layers.Dense(SEMANTIC_DIM, activation='linear', name='semantic')(x)
    return keras.Model(inputs, outputs)

def evaluate_zsl(model, X_test, y_test, unseen_classes, att_unseen):
    pred_sem = model.predict(X_test, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_unseen_norm = normalize(att_unseen, axis=1)
    scores = np.dot(pred_sem, att_unseen_norm.T)
    preds = [unseen_classes[i] for i in np.argmax(scores, axis=1)]
    return accuracy_score(y_test, preds)

def evaluate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all, all_classes, gamma):
    pred_sem = model.predict(X_gzsl, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_all_norm = normalize(att_all, axis=1)
    scores = np.dot(pred_sem, att_all_norm.T)
    seen_idx = [all_classes.index(c) for c in seen_classes]
    scores[:, seen_idx] += gamma
    preds = np.argmax(scores, axis=1)
    pred_c = np.array([all_classes[i] for i in preds])
    seen_mask = np.isin(y_gzsl, seen_classes)
    unseen_mask = np.isin(y_gzsl, unseen_classes)
    seen_acc = np.mean(pred_c[seen_mask] == y_gzsl[seen_mask]) if seen_mask.any() else 0.0
    unseen_acc = np.mean(pred_c[unseen_mask] == y_gzsl[unseen_mask]) if unseen_mask.any() else 0.0
    h = 2 * seen_acc * unseen_acc / (seen_acc + unseen_acc) if (seen_acc + unseen_acc) > 0 else 0.0
    return seen_acc, unseen_acc, h

def main():
    dataset_path = download_ucmerced()
    all_classes_full = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    X_raw, y_raw = extract_features(dataset_path, all_classes_full)
    X_all = preprocess_features(X_raw)
    glove = load_glove()

    # Original 14/7 split from search trial 37
    seen_classes = [
        'agricultural', 'airplane', 'beach', 'buildings', 'chaparral',
        'denseresidential', 'golfcourse', 'intersection', 'mobilehomepark',
        'overpass', 'river', 'runway', 'storagetanks', 'tenniscourt'
    ]
    unseen_classes = [
        'forest', 'freeway', 'mediumresidential', 'parkinglot', 'harbor',
        'baseballdiamond', 'sparseresidential'
    ]
    all_classes = sorted(seen_classes + unseen_classes)

    print("\n" + "="*60)
    print("SINGLE‑TASK SEMANTIC MAPPING (MSE only) – FIXED γ = -0.70")
    print("="*60)
    print(f"Seen classes ({len(seen_classes)}): {seen_classes}")
    print(f"Unseen classes ({len(unseen_classes)}): {unseen_classes}")

    word_emb = build_semantic_embeddings(all_classes, glove)
    att_matrix = np.stack([word_emb[c] for c in all_classes])
    att_seen = np.stack([word_emb[c] for c in seen_classes])

    scaler_att = StandardScaler()
    att_seen_white = scaler_att.fit_transform(att_seen)
    att_all_white = scaler_att.transform(att_matrix)
    att_unseen_white = scaler_att.transform(np.stack([word_emb[c] for c in unseen_classes]))

    seen_mask = np.isin(y_raw, seen_classes)
    unseen_mask = np.isin(y_raw, unseen_classes)
    X_seen = X_all[seen_mask]
    y_seen = y_raw[seen_mask]
    X_unseen = X_all[unseen_mask]
    y_unseen = y_raw[unseen_mask]
    print(f"Seen samples: {X_seen.shape[0]}, Unseen samples: {X_unseen.shape[0]}")

    y_seen_idx = np.array([seen_classes.index(c) for c in y_seen])
    train_att = att_seen_white[y_seen_idx]

    X_tr, X_val, A_tr, A_val = train_test_split(
        X_seen, train_att, test_size=0.2, random_state=SEED, stratify=y_seen_idx
    )

    model = build_semantic_model()
    model.summary()
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='mse',
        metrics=['mae']
    )

    callbacks = [
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=20, min_lr=1e-6, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True, verbose=1)
    ]

    print("\nTraining single‑task semantic mapping (MSE only)...")
    history = model.fit(
        X_tr, A_tr,
        validation_data=(X_val, A_val),
        batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks, verbose=1
    )

    final_val_loss = history.history['val_loss'][-1]
    print(f"\nFinal validation MSE loss: {final_val_loss:.4f}")

    zsl_acc = evaluate_zsl(model, X_unseen, y_unseen, unseen_classes, att_unseen_white)
    print(f"\nZSL Top-1: {zsl_acc:.4f}")

    # Build GZSL set (balanced 30 per class)
    X_gzsl, y_gzsl = [], []
    for c in seen_classes:
        idx = np.where(y_seen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_seen[idx])
            y_gzsl.extend([c] * len(idx))
    for c in unseen_classes:
        idx = np.where(y_unseen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_unseen[idx])
            y_gzsl.extend([c] * len(idx))
    X_gzsl = np.vstack(X_gzsl)
    y_gzsl = np.array(y_gzsl)

    # --- FIXED GAMMA = -0.70 ---
    GAMMA = -0.70
    seen_acc, unseen_acc, h = evaluate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes,
                                             att_all_white, all_classes, GAMMA)

    print("\n" + "="*50)
    print("RESULTS (fixed γ = -0.70)")
    print("="*50)
    print(f"Validation loss: {final_val_loss:.4f}")
    print(f"ZSL: {zsl_acc:.4f}")
    print(f"γ = {GAMMA:.2f} → Seen: {seen_acc:.4f}, Unseen: {unseen_acc:.4f}, H: {h:.4f}")

if __name__ == "__main__":
    main()

UCMerced already present.
Loading cached features...
Applying PCA and normalisation...
PCA: 2048 → 512-d (92.3% var)
Loaded GloVe from cache.

SINGLE‑TASK SEMANTIC MAPPING (MSE only) – FIXED γ = -0.70
Seen classes (14): ['agricultural', 'airplane', 'beach', 'buildings', 'chaparral', 'denseresidential', 'golfcourse', 'intersection', 'mobilehomepark', 'overpass', 'river', 'runway', 'storagetanks', 'tenniscourt']
Unseen classes (7): ['forest', 'freeway', 'mediumresidential', 'parkinglot', 'harbor', 'baseballdiamond', 'sparseresidential']
Seen samples: 1400, Unseen samples: 700


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 4096)           │     2,101,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 4096)           │        16,384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 2048)           │     8,390,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ semantic (Dense)                │ (None, 300)            │       614,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,131,180 (42.46 MB)

 Trainable params: 11,118,892 (42.42 MB)

 Non-trainable params: 12,288 (48.00 KB)


Training single‑task semantic mapping (MSE only)...
Epoch 1/500
9/9 ━━━━━━━━━━━━━━━━━━━━ 13s 644ms/step - loss: 1.9108 - mae: 1.0610 - val_loss: 0.9313 - val_mae: 0.7847 - learning_rate: 0.0030
Epoch 2/500
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 0.6612 - mae: 0.6387 - val_loss: 0.9026 - val_mae: 0.7729 - learning_rate: 0.0030
Epoch 3/500
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.4525 - mae: 0.5284 - val_loss: 0.8859 - val_mae: 0.7650 - learning_rate: 0.0030
Epoch 4/500
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.3663 - mae: 0.4736 - val_loss: 0.8839 - val_mae: 0.7634 - learning_rate: 0.0030
Epoch 5/500
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.3121 - mae: 0.4357 - val_loss: 0.8926 - val_mae: 0.7661 - learning_rate: 0.0030
Epoch 6/500
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.2827 - mae: 0.4138 - val_loss: 0.8984 - val_mae: 0.7679 - learning_rate: 0.0030
Epoch 7/500
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.2628 - mae: 0.3979 - val_loss: 0.9019 - val_m

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
UCMerced GZSL – Pure MSE, deep network, cosine LR decay.
Target: seen >80%, unseen >40%, val loss <0.1
"""

import os
import pickle
import random
import zipfile
import warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import normalize, MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import wordnet as wn

warnings.filterwarnings('ignore')
SEED = 3483
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

VISUAL_DIM = 512
SEMANTIC_DIM = 300
HIDDEN_DIM = 12288              # very wide
DROPOUT_RATE = 0.4              # strong dropout
BATCH_SIZE = 512
EPOCHS = 1500
INIT_LR = 1e-3
FEATURE_CACHE = "ucmerced_resnet101_features.npz"
GLOVE_CACHE = "glove_embeddings.pkl"

# -------------------------- Data utilities (unchanged) --------------------------
def download_ucmerced():
    dest = "/content/UCMerced"
    img_dir = os.path.join(dest, "UCMerced_LandUse", "Images")
    if os.path.isdir(img_dir):
        print("UCMerced already present.")
        return img_dir
    print("Downloading UCMerced...")
    os.system("pip install -q gdown")
    gdrive_id = "1J27m1nFMfqRNYIPNRTIgbBEynI8JJW8U"
    os.system(f"gdown --id {gdrive_id} -O /tmp/ucmerced.zip 2>/dev/null")
    if not os.path.exists("/tmp/ucmerced.zip") or os.path.getsize("/tmp/ucmerced.zip") < 1e6:
        try:
            from google.colab import files
            if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
                print("Upload kaggle.json...")
                files.upload()
                os.system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
        except:
            pass
        os.system("kaggle datasets download -d abdulhasibuddin/uc-merced-land-use-dataset -p /tmp/ --quiet && mv /tmp/uc-merced-land-use-dataset.zip /tmp/ucmerced.zip")
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile("/tmp/ucmerced.zip", 'r') as z:
        z.extractall(dest)
    os.remove("/tmp/ucmerced.zip")
    for candidate in [os.path.join(dest, "UCMerced_LandUse", "Images"), os.path.join(dest, "Images")]:
        if os.path.isdir(candidate):
            print(f"UCMerced ready at {candidate}")
            return candidate
    raise FileNotFoundError("Could not find Images folder.")

def extract_features(dataset_path, class_names):
    if os.path.exists(FEATURE_CACHE):
        print("Loading cached features...")
        data = np.load(FEATURE_CACHE, allow_pickle=True)
        return data['X'].astype('float32'), data['y'].astype(str)
    print("Extracting ResNet101 features...")
    base = tf.keras.applications.ResNet101(include_top=False, weights='imagenet',
                                           input_shape=(224,224,3), pooling='avg')
    base.trainable = False
    inp = tf.keras.Input(shape=(224,224,3))
    out = base(tf.keras.applications.resnet.preprocess_input(inp), training=False)
    extractor = tf.keras.Model(inp, out)
    all_X, all_y = [], []
    for cls in class_names:
        cls_dir = os.path.join(dataset_path, cls)
        if not os.path.isdir(cls_dir):
            continue
        imgs = []
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(('.jpg','.png','.jpeg','.tif','.tiff')):
                continue
            try:
                img = tf.keras.utils.load_img(os.path.join(cls_dir, fname), target_size=(224,224))
                imgs.append(tf.keras.utils.img_to_array(img))
            except:
                continue
        if imgs:
            feats = extractor.predict(np.stack(imgs), batch_size=32, verbose=0).astype('float32')
            all_X.append(feats)
            all_y.extend([cls] * len(feats))
            print(f"  {cls}: {len(feats)} images")
    del extractor
    keras.backend.clear_session()
    X = np.concatenate(all_X, axis=0)
    y = np.array(all_y)
    np.savez_compressed(FEATURE_CACHE, X=X, y=y)
    print(f"Features cached: {X.shape}")
    return X, y

def preprocess_features(X_raw):
    print("Applying PCA and normalisation...")
    X = normalize(MinMaxScaler().fit_transform(X_raw), axis=1)
    pca = PCA(n_components=VISUAL_DIM, random_state=SEED)
    X_pca = pca.fit_transform(X).astype('float32')
    X_pca = normalize(X_pca, axis=1)
    print(f"PCA: {X_raw.shape[1]} → {VISUAL_DIM}-d ({100*pca.explained_variance_ratio_.sum():.1f}% var)")
    return X_pca

def load_glove():
    if os.path.exists(GLOVE_CACHE):
        with open(GLOVE_CACHE, "rb") as f:
            glove = pickle.load(f)
        print("Loaded GloVe from cache.")
        return glove
    if not os.path.exists("glove.6B.300d.txt"):
        print("Downloading GloVe...")
        os.system("wget -q https://nlp.stanford.edu/data/glove.6B.zip -O glove.6B.zip")
        os.system("unzip -q glove.6B.zip glove.6B.300d.txt")
        os.system("rm -f glove.6B.zip")
    print("Loading GloVe from text file...")
    glove = {}
    with open("glove.6B.300d.txt", 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            vec = np.asarray(parts[1:], dtype='float32')
            if len(vec) == 300:
                glove[word] = vec
    with open(GLOVE_CACHE, "wb") as f:
        pickle.dump(glove, f)
    print(f"Loaded {len(glove)} vectors.")
    return glove

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def augment_semantic_vector(class_name, glove, top_n=5):
    words = class_name.replace('_', ' ').split()
    primary_vecs = [glove[w] for w in words if w in glove]
    if not primary_vecs:
        primary_vecs = [np.random.randn(300).astype('float32') * 0.01]
    main_vec = np.mean(primary_vecs, axis=0)
    syn_vecs = []
    for syn in wn.synsets(class_name.replace('_', ' ')):
        for lemma in syn.lemmas():
            lemma_name = lemma.name().replace('_', ' ')
            if lemma_name != class_name.replace('_', ' ') and lemma_name in glove:
                syn_vecs.append(glove[lemma_name])
    syn_vecs = list(set([tuple(v) for v in syn_vecs]))
    if syn_vecs:
        n = min(top_n, len(syn_vecs))
        chosen = np.array(random.sample(syn_vecs, n))
        avg_syn = np.mean(chosen, axis=0)
        augmented = (main_vec + avg_syn) / 2
    else:
        augmented = main_vec
    return augmented / (np.linalg.norm(augmented) + 1e-8)

def build_semantic_embeddings(class_names, glove):
    return {cls: augment_semantic_vector(cls, glove) for cls in class_names}

# ---------- PURE MSE MODEL (no auxiliary classifier) ----------
def build_pure_mse_model():
    inputs = keras.Input(shape=(VISUAL_DIM,))
    x = layers.Dense(HIDDEN_DIM, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(HIDDEN_DIM//2, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    x = layers.Dense(HIDDEN_DIM//4, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    outputs = layers.Dense(SEMANTIC_DIM, activation='linear', name='semantic')(x)
    return keras.Model(inputs, outputs)

def evaluate_zsl(model, X_test, y_test, unseen_classes, att_unseen):
    pred_sem = model.predict(X_test, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_unseen_norm = normalize(att_unseen, axis=1)
    scores = np.dot(pred_sem, att_unseen_norm.T)
    preds = [unseen_classes[i] for i in np.argmax(scores, axis=1)]
    return accuracy_score(y_test, preds)

def evaluate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes, att_all, all_classes, gamma):
    pred_sem = model.predict(X_gzsl, batch_size=128, verbose=0)
    pred_sem = normalize(pred_sem, axis=1)
    att_all_norm = normalize(att_all, axis=1)
    scores = np.dot(pred_sem, att_all_norm.T)
    seen_idx = [all_classes.index(c) for c in seen_classes]
    scores[:, seen_idx] += gamma
    preds = np.argmax(scores, axis=1)
    pred_c = np.array([all_classes[i] for i in preds])
    seen_mask = np.isin(y_gzsl, seen_classes)
    unseen_mask = np.isin(y_gzsl, unseen_classes)
    seen_acc = np.mean(pred_c[seen_mask] == y_gzsl[seen_mask]) if seen_mask.any() else 0.0
    unseen_acc = np.mean(pred_c[unseen_mask] == y_gzsl[unseen_mask]) if unseen_mask.any() else 0.0
    h = 2 * seen_acc * unseen_acc / (seen_acc + unseen_acc) if (seen_acc + unseen_acc) > 0 else 0.0
    return seen_acc, unseen_acc, h

def main():
    dataset_path = download_ucmerced()
    all_classes_full = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    X_raw, y_raw = extract_features(dataset_path, all_classes_full)
    X_all = preprocess_features(X_raw)
    glove = load_glove()

    # Fixed 14/7 split
    seen_classes = [
        'agricultural', 'airplane', 'beach', 'buildings', 'chaparral',
        'denseresidential', 'golfcourse', 'intersection', 'mobilehomepark',
        'overpass', 'river', 'runway', 'storagetanks', 'tenniscourt'
    ]
    unseen_classes = [
        'forest', 'freeway', 'mediumresidential', 'parkinglot', 'harbor',
        'baseballdiamond', 'sparseresidential'
    ]
    all_classes = sorted(seen_classes + unseen_classes)

    print("\n" + "="*60)
    print("PURE MSE MODEL (no auxiliary classifier) – Deep wide network")
    print("="*60)
    print(f"Seen classes (14): {seen_classes}")
    print(f"Unseen classes (7): {unseen_classes}")

    word_emb = build_semantic_embeddings(all_classes, glove)
    att_matrix = np.stack([word_emb[c] for c in all_classes])
    att_seen = np.stack([word_emb[c] for c in seen_classes])

    scaler_att = StandardScaler()
    att_seen_white = scaler_att.fit_transform(att_seen)
    att_all_white = scaler_att.transform(att_matrix)
    att_unseen_white = scaler_att.transform(np.stack([word_emb[c] for c in unseen_classes]))

    seen_mask = np.isin(y_raw, seen_classes)
    unseen_mask = np.isin(y_raw, unseen_classes)
    X_seen = X_all[seen_mask]
    y_seen = y_raw[seen_mask]
    X_unseen = X_all[unseen_mask]
    y_unseen = y_raw[unseen_mask]
    print(f"Seen samples: {X_seen.shape[0]}, Unseen samples: {X_unseen.shape[0]}")

    y_seen_idx = np.array([seen_classes.index(c) for c in y_seen])
    train_att = att_seen_white[y_seen_idx]

    X_tr, X_val, A_tr, A_val = train_test_split(
        X_seen, train_att, test_size=0.2, random_state=SEED, stratify=y_seen_idx
    )

    model = build_pure_mse_model()
    model.summary()

    # Cosine decay learning rate schedule
    total_steps = (len(X_tr) // BATCH_SIZE) * EPOCHS
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=INIT_LR,
        decay_steps=total_steps,
        alpha=1e-3  # final learning rate = INIT_LR * alpha = 1e-6
    )
    optimizer = keras.optimizers.Adam(learning_rate=lr_schedule)

    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

    # Only early stopping – LR schedule already decays, so no ReduceLROnPlateau
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=80, restore_best_weights=True, verbose=1, mode='min')
    ]

    print("\nTraining pure MSE model (no auxiliary loss)...")
    history = model.fit(
        X_tr, A_tr,
        validation_data=(X_val, A_val),
        batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks, verbose=1
    )

    final_val_loss = history.history['val_loss'][-1]
    print(f"\nFinal validation MSE loss: {final_val_loss:.4f}")

    zsl_acc = evaluate_zsl(model, X_unseen, y_unseen, unseen_classes, att_unseen_white)
    print(f"\nZSL Top-1: {zsl_acc:.4f}")

    # Build GZSL set (balanced 30 per class)
    X_gzsl, y_gzsl = [], []
    for c in seen_classes:
        idx = np.where(y_seen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_seen[idx])
            y_gzsl.extend([c] * len(idx))
    for c in unseen_classes:
        idx = np.where(y_unseen == c)[0][:30]
        if len(idx) > 0:
            X_gzsl.append(X_unseen[idx])
            y_gzsl.extend([c] * len(idx))
    X_gzsl = np.vstack(X_gzsl)
    y_gzsl = np.array(y_gzsl)

    # Search gamma that meets both targets
    print("\nSearching gamma for seen>80%, unseen>40%...")
    best_gamma = None
    best_seen = best_unseen = best_h = 0
    target_gamma = None
    for gamma in np.arange(-1.5, 2.0, 0.05):
        seen_acc, unseen_acc, h = evaluate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes,
                                                 att_all_white, all_classes, gamma)
        if unseen_acc > 0.40 and seen_acc > 0.80:
            target_gamma = gamma
            best_seen = seen_acc
            best_unseen = unseen_acc
            best_h = h
            break
        if h > best_h:
            best_h = h
            best_gamma = gamma
            best_seen = seen_acc
            best_unseen = unseen_acc

    print("\n" + "="*50)
    print("FINAL RESULTS")
    print("="*50)
    print(f"Validation MSE loss: {final_val_loss:.4f}")
    print(f"ZSL: {zsl_acc:.4f}")
    if target_gamma is not None:
        print(f"\n✓ Gamma = {target_gamma:.2f} achieves both targets:")
        print(f"  Seen: {best_seen:.4f} (>80%)   Unseen: {best_unseen:.4f} (>40%)   H: {best_h:.4f}")
    else:
        print(f"\n⚠️ No gamma found with both seen>80% and unseen>40%.")
        print(f"Best H-score gamma = {best_gamma:.2f} → Seen: {best_seen:.4f}, Unseen: {best_unseen:.4f}, H: {best_h:.4f}")
        if final_val_loss < 0.1:
            print("Validation loss OK. Try expanding gamma range or using different gamma step.")
        else:
            print("Validation loss too high. Increase model capacity or train longer.")

    # Reference gamma = -0.70
    s70, u70, h70 = evaluate_gamma(model, X_gzsl, y_gzsl, seen_classes, unseen_classes,
                                   att_all_white, all_classes, -0.70)
    print(f"\nReference (γ = -0.70) → Seen: {s70:.4f}, Unseen: {u70:.4f}, H: {h70:.4f}")

if __name__ == "__main__":
    main()

UCMerced already present.
Loading cached features...
Applying PCA and normalisation...
PCA: 2048 → 512-d (92.3% var)
Loaded GloVe from cache.

PURE MSE MODEL (no auxiliary classifier) – Deep wide network
Seen classes (14): ['agricultural', 'airplane', 'beach', 'buildings', 'chaparral', 'denseresidential', 'golfcourse', 'intersection', 'mobilehomepark', 'overpass', 'river', 'runway', 'storagetanks', 'tenniscourt']
Unseen classes (7): ['forest', 'freeway', 'mediumresidential', 'parkinglot', 'harbor', 'baseballdiamond', 'sparseresidential']
Seen samples: 1400, Unseen samples: 700


Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 12288)          │     6,303,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 12288)          │        49,152 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25 (Dropout)            │ (None, 12288)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 6144)           │    75,503,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_26          │ (None, 6144)           │        24,576 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 6144)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 3072)           │    18,877,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_27          │ (None, 3072)           │        12,288 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ semantic (Dense)                │ (None, 300)            │       921,900 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,692,716 (387.93 MB)

 Trainable params: 101,649,708 (387.76 MB)

 Non-trainable params: 43,008 (168.00 KB)


Training pure MSE model (no auxiliary loss)...
Epoch 1/1500
3/3 ━━━━━━━━━━━━━━━━━━━━ 14s 3s/step - loss: 4.0119 - mae: 1.5880 - val_loss: 0.9822 - val_mae: 0.8060
Epoch 2/1500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 353ms/step - loss: 2.6268 - mae: 1.2764 - val_loss: 0.9562 - val_mae: 0.7955
Epoch 3/1500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 327ms/step - loss: 1.9613 - mae: 1.0952 - val_loss: 0.9322 - val_mae: 0.7854
Epoch 4/1500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 366ms/step - loss: 1.6157 - mae: 0.9917 - val_loss: 0.9151 - val_mae: 0.7782
Epoch 5/1500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 366ms/step - loss: 1.4651 - mae: 0.9381 - val_loss: 0.8893 - val_mae: 0.7671
Epoch 6/1500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 368ms/step - loss: 1.3529 - mae: 0.8993 - val_loss: 0.8702 - val_mae: 0.7587
Epoch 7/1500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 320ms/step - loss: 1.2730 - mae: 0.8685 - val_loss: 0.8632 - val_mae: 0.7553
Epoch 8/1500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 307ms/step - loss: 1.2021 - mae: 0.8426 - val_loss: 0.8592 - val_mae: 0.7534
Epoch 9/15